In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor


# ============================================================
# Random seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# Load data
# ============================================================

# data = pd.read_csv(
#     "../../../data/processed/model_df_next_mmse_blood.csv"
# )

data = pd.read_csv(
    "../../../data/processed/model_df_next_mmse3.csv"
)

print(data.shape)

df = data.copy()


#### 缺失資料

In [ ]:
# ============================================================
# Missing value summary
# ============================================================

missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_rate_pct": data.isna().mean() * 100
})

# 只保留有缺失值的欄位
missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

# 缺失比例由大到小排序
missing_summary = missing_summary.sort_values(
    "missing_rate_pct",
    ascending=False
)

# 百分比四捨五入
missing_summary["missing_rate_pct"] = (
    missing_summary["missing_rate_pct"]
    .round(2)
)

print("\n================ Missing Value Summary ================\n")
print(missing_summary)

print("\n有缺失值的欄位數：", len(missing_summary))
print("總欄位數：", data.shape[1])
print("總資料筆數：", data.shape[0])

### 處理日期並排序 / 建立 visit gap days欄位，代表每次MMSE量測日期的時間差距

In [ ]:
df["visit_date"] = pd.to_datetime(
    df["visit_date"],
    errors="coerce"
)

df = df.sort_values(
    ["patient_id", "visit_date"]
).reset_index(drop=True)

df["visit_gap_days"] = (
    df.groupby("patient_id")["visit_date"]
    .diff()
    .dt.days
)
#第一次 visit 沒有上一筆，所以填 0
df["visit_gap_days"] = (
    df["visit_gap_days"]
    .fillna(0)
)
#因為天數可能非常大，所以做：log (1+gap)
df["visit_gap_log"] = np.log1p(
    df["visit_gap_days"].clip(lower=0)
)

In [ ]:
target_history_features = [
    "mmse"
]

cdr_features = [
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob",
]

casi_features = [
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total",
    "days_since_casi",
    "has_recent_casi",
]

blood_features = [
    "hdl",
    "vldl",
    "ldl",
    "total_cholesterol",
    "triglyceride",
    "fasting_glucose",
    "hba1c",
]

blood_gap_features = [
    "hdl_gap_days",
    "vldl_gap_days",
    "ldl_gap_days",
    "total_cholesterol_gap_days",
    "triglyceride_gap_days",
    "fasting_glucose_gap_days",
    "hba1c_gap_days",
]

#這些 血液日期gap 可能很大，所以也做 log transform
for col in blood_gap_features:

    new_col = col + "_log"

    df[new_col] = np.log1p(
        df[col].clip(lower=0)
    )

blood_gap_log_features = [
    col + "_log"
    for col in blood_gap_features
]

#其他 clinical features
clinical_features = [
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",
    "apoe4_count",
    "apoe_missing",
    "gender"
]

#時間
time_features = [
    "visit_gap_log"
]

#全部
feature_cols = (
    target_history_features
    + cdr_features
    + casi_features
    + blood_features
    + blood_gap_log_features
    + clinical_features
    + time_features
)

print("Feature 數量:", len(feature_cols))

for col in feature_cols:
    print(col)

In [ ]:
visit_counts = (
    df.groupby("patient_id")
    .size()
)

visit_counts.describe()

### 建立df_model資料集，保留至少兩次mmse visit的病人

In [ ]:
valid_patient_ids = (
    visit_counts[
        visit_counts >= 2
    ]
    .index
)

df_model = df[
    df["patient_id"].isin(valid_patient_ids)
].copy()

print(
    "至少兩次 visit 的 patient:",
    df_model["patient_id"].nunique()
)

In [ ]:
patient_group_counts = (
    df_model[
        ["patient_id", "bio_category"]
    ]
    .drop_duplicates()
    ["bio_category"]
    .value_counts()
)

print(patient_group_counts)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_model.isnull().sum(),
    "missing_ratio": df_model.isnull().mean()
})

missing_summary = missing_summary[missing_summary["missing_count"] > 0] \
    .sort_values("missing_count", ascending=False)

print(missing_summary)



In [ ]:
casi_missing_patient_count = (
    df_model.loc[df_model["casi_short_term_memory"].isnull(), "patient_id"]
    .nunique()
)

fasting_glucose_missing_patient_count = (
    df_model.loc[df_model["fasting_glucose"].isnull(), "patient_id"]
    .nunique()
)

print("casi_short_term_memory 缺失的 patient_id 數量：", casi_missing_patient_count)
print("fasting_glucose 缺失的 patient_id 數量：", fasting_glucose_missing_patient_count)

In [ ]:
casi_all_missing_patient_count = (
    df_model.groupby("patient_id")["casi_short_term_memory"]
    .apply(lambda x: x.isnull().all())
    .sum()
)

fasting_glucose_all_missing_patient_count = (
    df_model.groupby("patient_id")["fasting_glucose"]
    .apply(lambda x: x.isnull().all())
    .sum()
)

print("所有紀錄的 casi_short_term_memory 都缺失的 patient_id 數量：", casi_all_missing_patient_count)
print("所有紀錄的 fasting_glucose 都缺失的 patient_id 數量：", fasting_glucose_all_missing_patient_count)

### 模型訓練

## CU / AD / non-AD 分組建模與總結


In [ ]:
all_group_results = []
all_group_predictions = []

# 儲存 CU / AD / non-AD 各模型的 feature importance
all_feature_importance = []


# Group: CU


In [ ]:
from sklearn.model_selection import train_test_split

GROUP = "CU"

group_df = df_model[
    df_model["bio_category"] == GROUP
].copy()

print(group_df.shape)

print(
    "Patients:",
    group_df["patient_id"].nunique()
)

#Patient-level train/validation/test split
patient_ids = (
    group_df["patient_id"]
    .drop_duplicates()
    .to_numpy()
)

train_ids, temp_ids = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=SEED
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=SEED
)

print("Train patients:", len(train_ids))
print("Val patients:", len(val_ids))
print("Test patients:", len(test_ids))

print(
    set(train_ids)
    & set(val_ids)
)

print(
    set(train_ids)
    & set(test_ids)
)

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

### 建立missing indicators

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

missing_indicator_cols = []

for col in feature_cols:

    if train_df[col].isna().any():

        indicator_col = col + "_missing"

        train_df[indicator_col] = (
            train_df[col].isna().astype(int)
        )

        val_df[indicator_col] = (
            val_df[col].isna().astype(int)
        )

        test_df[indicator_col] = (
            test_df[col].isna().astype(int)
        )

        missing_indicator_cols.append(
            indicator_col
        )

In [ ]:
print(
    "Missing indicators:",
    len(missing_indicator_cols)
)

### 缺值處理 (中位數填補)

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)

imputer.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = imputer.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = imputer.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = imputer.transform(
    test_df[feature_cols]
)

In [ ]:
final_feature_cols = (
    feature_cols
    + missing_indicator_cols
)

### StandardScaler

SVM 對特徵尺度非常敏感，因此 numeric features 使用 train set fit 的 StandardScaler。
Random Forest / XGBoost 對尺度不敏感，但為了讓三個模型共用完全相同的輸入資料，
這裡仍使用同一份標準化後的資料；missing indicator 保持 0/1，不做 scaling。

In [ ]:
scaler = StandardScaler()
# missing indicator 不需要 scaling。所以只 scale 原始數值：
scaler.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = scaler.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = scaler.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = scaler.transform(
    test_df[feature_cols]
)

### 將 longitudinal history 轉成固定長度特徵

SVM、Random Forest、XGBoost 無法直接處理每位病人長度不同的 sequence，
因此對「target 前的所有歷史 visits」建立固定長度 summary features：

- `last`：最近一次歷史 visit
- `mean`：歷史平均
- `std`：歷史標準差
- `min` / `max`：歷史範圍
- `history_n_visits`：target 前共有幾次 visit
- `history_span_days`：第一個歷史 visit 到最近歷史 visit 的天數

**Target 仍然是該病人的最後一次 MMSE，因此 test target 沒有被拿來建立輸入特徵。**

In [ ]:
def create_tabular_history_features(
    data,
    feature_cols
):
    """
    將每位 patient 在 target visit 之前的 longitudinal history
    轉成固定長度向量，供 SVM / RF / XGBoost 使用。

    Target = 該 patient 最後一次 visit 的 MMSE
    History = 最後一次 visit 之前的所有 visits
    """

    X_rows = []
    y = []
    patient_ids = []

    for patient_id, patient_df in data.groupby("patient_id"):

        patient_df = (
            patient_df
            .sort_values("visit_date")
            .reset_index(drop=True)
        )

        # 至少要有 2 次 visit：
        # 前面 >= 1 次做 history，最後 1 次做 target
        if len(patient_df) < 2:
            continue

        history = patient_df.iloc[:-1].copy()
        target_visit = patient_df.iloc[-1]

        hist_values = history[feature_cols]

        row = {}

        # 最近一次歷史 visit
        last_values = hist_values.iloc[-1]
        for col in feature_cols:
            row[f"{col}_last"] = last_values[col]

        # 使用全部歷史 visit 的 summary
        means = hist_values.mean(axis=0)
        stds = hist_values.std(axis=0, ddof=0)
        mins = hist_values.min(axis=0)
        maxs = hist_values.max(axis=0)

        for col in feature_cols:
            row[f"{col}_mean"] = means[col]
            row[f"{col}_std"] = stds[col]
            row[f"{col}_min"] = mins[col]
            row[f"{col}_max"] = maxs[col]

        row["history_n_visits"] = len(history)

        if len(history) >= 2:
            row["history_span_days"] = (
                history["visit_date"].iloc[-1]
                - history["visit_date"].iloc[0]
            ).days
        else:
            row["history_span_days"] = 0

        X_rows.append(row)
        y.append(float(target_visit["mmse"]))
        patient_ids.append(patient_id)

    X = pd.DataFrame(X_rows)

    return (
        X,
        np.asarray(y, dtype=float),
        np.asarray(patient_ids)
    )


In [ ]:
X_train, y_train, train_patient_ids = (
    create_tabular_history_features(
        train_df,
        final_feature_cols
    )
)

X_val, y_val, val_patient_ids = (
    create_tabular_history_features(
        val_df,
        final_feature_cols
    )
)

X_test, y_test, test_patient_ids = (
    create_tabular_history_features(
        test_df,
        final_feature_cols
    )
)

# 保證三個 split 欄位順序完全一致
X_val = X_val.reindex(columns=X_train.columns)
X_test = X_test.reindex(columns=X_train.columns)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("Train patients:", len(train_patient_ids))
print("Val patients  :", len(val_patient_ids))
print("Test patients :", len(test_patient_ids))


### 模型與 validation tuning

三個模型都只用 validation set 選超參數，**test set 不參與選模**。
選好參數後，再使用 `train + validation` 重新 fit，最後只在 test set 評估一次。

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def tune_on_validation(
    estimator_class,
    param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    fixed_params=None
):
    """
    使用 validation RMSE 選擇最佳超參數。
    """
    fixed_params = fixed_params or {}

    tuning_rows = []
    best_rmse = np.inf
    best_params = None

    for params in ParameterGrid(param_grid):

        model = estimator_class(
            **fixed_params,
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        val_pred = model.predict(
            X_val
        )

        metrics = regression_metrics(
            y_val,
            val_pred
        )

        tuning_rows.append({
            **params,
            **metrics
        })

        if metrics["RMSE"] < best_rmse:
            best_rmse = metrics["RMSE"]
            best_params = params.copy()

    tuning_df = (
        pd.DataFrame(tuning_rows)
        .sort_values("RMSE")
        .reset_index(drop=True)
    )

    return best_params, tuning_df


### 1. SVM（SVR）

In [ ]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.1, 0.5, 1.0],
    "gamma": ["scale", 0.01, 0.1]
}

best_svm_params, svm_tuning = tune_on_validation(
    estimator_class=SVR,
    param_grid=svm_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "kernel": "rbf"
    }
)

print("Best SVM params:")
print(best_svm_params)

display(
    svm_tuning.head(10)
)


### 2. Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5]
}

best_rf_params, rf_tuning = tune_on_validation(
    estimator_class=RandomForestRegressor,
    param_grid=rf_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "random_state": SEED,
        "n_jobs": -1
    }
)

print("Best RF params:")
print(best_rf_params)

display(
    rf_tuning.head(10)
)


### 3. XGBoost

In [ ]:
xgb_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0]
}

best_xgb_params, xgb_tuning = tune_on_validation(
    estimator_class=XGBRegressor,
    param_grid=xgb_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "objective": "reg:squarederror",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0
    }
)

print("Best XGBoost params:")
print(best_xgb_params)

display(
    xgb_tuning.head(10)
)


### 使用最佳參數重新訓練，並在 test set 比較

Validation 已完成選模後，將 train + validation 合併重新訓練。
Test set 在這之前完全沒有參與 tuning。

In [ ]:
X_trainval = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_trainval = np.concatenate(
    [y_train, y_val]
)


svm_model = SVR(
    kernel="rbf",
    **best_svm_params
)

rf_model = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
    **best_rf_params
)

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
    **best_xgb_params
)


models = {
    "SVM": svm_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

test_predictions = {}
result_rows = []

for model_name, model in models.items():

    model.fit(
        X_trainval,
        y_trainval
    )

    y_pred = model.predict(
        X_test
    )

    test_predictions[model_name] = y_pred

    metrics = regression_metrics(
        y_test,
        y_pred
    )

    result_rows.append({
        "Model": model_name,
        **metrics
    })


results_df = (
    pd.DataFrame(result_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(results_df)


In [ ]:
# ============================================================
# Feature importance：Permutation Importance
# ============================================================
# 使用同一種 permutation importance 計算 SVM / RF / XGBoost，
# 因此三種模型的重要性可用一致定義比較。
# scoring 使用 negative RMSE；importance > 0 代表打亂該特徵會使預測變差。

feature_names = list(X_test.columns)

group_importance_rows = []

for model_name, model in models.items():
    perm = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="neg_root_mean_squared_error",
        n_repeats=20,
        random_state=SEED,
        n_jobs=-1
    )

    temp_imp = pd.DataFrame({
        "Group": GROUP,
        "Model": model_name,
        "Feature": feature_names,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std,
        "Method": "Permutation importance (test RMSE)"
    })

    temp_imp = temp_imp.sort_values(
        "Importance", ascending=False
    ).reset_index(drop=True)
    temp_imp["Rank"] = np.arange(1, len(temp_imp) + 1)

    group_importance_rows.append(temp_imp)
    all_feature_importance.append(temp_imp)

    print(f"\n=== {GROUP} | {model_name} | Top 20 features ===")
    display(temp_imp.head(20))

group_feature_importance = pd.concat(
    group_importance_rows, ignore_index=True
)


In [ ]:
# 每位 test patient 的實際值與三個模型預測值
prediction_df = pd.DataFrame({
    "patient_id": test_patient_ids,
    "y_true": y_test,
    "SVM_pred": test_predictions["SVM"],
    "RF_pred": test_predictions["Random Forest"],
    "XGBoost_pred": test_predictions["XGBoost"]
})

display(
    prediction_df.head()
)


In [ ]:
# Actual vs Predicted
for model_name, y_pred in test_predictions.items():

    plt.figure(figsize=(6, 6))

    plt.scatter(
        y_test,
        y_pred,
        alpha=0.7
    )

    lower = min(
        y_test.min(),
        y_pred.min()
    )
    upper = max(
        y_test.max(),
        y_pred.max()
    )

    plt.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--"
    )

    plt.xlabel("True MMSE")
    plt.ylabel("Predicted MMSE")
    plt.title(f"{model_name}: True vs Predicted")
    plt.show()


### Last-MMSE baseline

In [ ]:
def last_mmse_baseline(
    data,
    patient_ids
):

    predictions = []
    targets = []

    for patient_id in patient_ids:

        patient_df = (
            data[
                data["patient_id"]
                == patient_id
            ]
            .sort_values(
                "visit_date"
            )
        )

        if len(patient_df) < 2:
            continue

        previous_mmse = (
            patient_df.iloc[-2]["mmse"]
        )

        target_mmse = (
            patient_df.iloc[-1]["mmse"]
        )

        predictions.append(
            previous_mmse
        )

        targets.append(
            target_mmse
        )

    return (
        np.array(predictions),
        np.array(targets)
    )

In [ ]:
baseline_pred, baseline_true = (
    last_mmse_baseline(
        test_df,
        test_ids
    )
)

baseline_mae = mean_absolute_error(
    baseline_true,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_true,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    baseline_true,
    baseline_pred
)

print("Last-MMSE baseline")

print(
    f"MAE : {baseline_mae:.3f}"
)

print(
    f"RMSE: {baseline_rmse:.3f}"
)

print(
    f"R²  : {baseline_r2:.3f}"
)

In [ ]:
# 儲存此組別的 test 結果
group_result = results_df.copy()
group_result.insert(0, "Group", GROUP)

baseline_row = pd.DataFrame([{
    "Group": GROUP,
    "Model": "Last-MMSE baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
}])
# 對齊實際 metric 欄名（R2 或 R²）
if "R²" in group_result.columns and "R2" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R2": "R²"})
elif "R2" in group_result.columns and "R²" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R²": "R2"})

all_group_results.append(pd.concat([group_result, baseline_row], ignore_index=True))

group_pred = prediction_df.copy()
group_pred.insert(0, "Group", GROUP)
all_group_predictions.append(group_pred)


# Group: AD


In [ ]:
from sklearn.model_selection import train_test_split

GROUP = "AD"

group_df = df_model[
    df_model["bio_category"] == GROUP
].copy()

print(group_df.shape)

print(
    "Patients:",
    group_df["patient_id"].nunique()
)

#Patient-level train/validation/test split
patient_ids = (
    group_df["patient_id"]
    .drop_duplicates()
    .to_numpy()
)

train_ids, temp_ids = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=SEED
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=SEED
)

print("Train patients:", len(train_ids))
print("Val patients:", len(val_ids))
print("Test patients:", len(test_ids))

print(
    set(train_ids)
    & set(val_ids)
)

print(
    set(train_ids)
    & set(test_ids)
)

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

### 建立missing indicators

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

missing_indicator_cols = []

for col in feature_cols:

    if train_df[col].isna().any():

        indicator_col = col + "_missing"

        train_df[indicator_col] = (
            train_df[col].isna().astype(int)
        )

        val_df[indicator_col] = (
            val_df[col].isna().astype(int)
        )

        test_df[indicator_col] = (
            test_df[col].isna().astype(int)
        )

        missing_indicator_cols.append(
            indicator_col
        )

In [ ]:
print(
    "Missing indicators:",
    len(missing_indicator_cols)
)

### 缺值處理 (中位數填補)

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)

imputer.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = imputer.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = imputer.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = imputer.transform(
    test_df[feature_cols]
)

In [ ]:
final_feature_cols = (
    feature_cols
    + missing_indicator_cols
)

### StandardScaler

SVM 對特徵尺度非常敏感，因此 numeric features 使用 train set fit 的 StandardScaler。
Random Forest / XGBoost 對尺度不敏感，但為了讓三個模型共用完全相同的輸入資料，
這裡仍使用同一份標準化後的資料；missing indicator 保持 0/1，不做 scaling。

In [ ]:
scaler = StandardScaler()
# missing indicator 不需要 scaling。所以只 scale 原始數值：
scaler.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = scaler.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = scaler.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = scaler.transform(
    test_df[feature_cols]
)

### 將 longitudinal history 轉成固定長度特徵

SVM、Random Forest、XGBoost 無法直接處理每位病人長度不同的 sequence，
因此對「target 前的所有歷史 visits」建立固定長度 summary features：

- `last`：最近一次歷史 visit
- `mean`：歷史平均
- `std`：歷史標準差
- `min` / `max`：歷史範圍
- `history_n_visits`：target 前共有幾次 visit
- `history_span_days`：第一個歷史 visit 到最近歷史 visit 的天數

**Target 仍然是該病人的最後一次 MMSE，因此 test target 沒有被拿來建立輸入特徵。**

In [ ]:
def create_tabular_history_features(
    data,
    feature_cols
):
    """
    將每位 patient 在 target visit 之前的 longitudinal history
    轉成固定長度向量，供 SVM / RF / XGBoost 使用。

    Target = 該 patient 最後一次 visit 的 MMSE
    History = 最後一次 visit 之前的所有 visits
    """

    X_rows = []
    y = []
    patient_ids = []

    for patient_id, patient_df in data.groupby("patient_id"):

        patient_df = (
            patient_df
            .sort_values("visit_date")
            .reset_index(drop=True)
        )

        # 至少要有 2 次 visit：
        # 前面 >= 1 次做 history，最後 1 次做 target
        if len(patient_df) < 2:
            continue

        history = patient_df.iloc[:-1].copy()
        target_visit = patient_df.iloc[-1]

        hist_values = history[feature_cols]

        row = {}

        # 最近一次歷史 visit
        last_values = hist_values.iloc[-1]
        for col in feature_cols:
            row[f"{col}_last"] = last_values[col]

        # 使用全部歷史 visit 的 summary
        means = hist_values.mean(axis=0)
        stds = hist_values.std(axis=0, ddof=0)
        mins = hist_values.min(axis=0)
        maxs = hist_values.max(axis=0)

        for col in feature_cols:
            row[f"{col}_mean"] = means[col]
            row[f"{col}_std"] = stds[col]
            row[f"{col}_min"] = mins[col]
            row[f"{col}_max"] = maxs[col]

        row["history_n_visits"] = len(history)

        if len(history) >= 2:
            row["history_span_days"] = (
                history["visit_date"].iloc[-1]
                - history["visit_date"].iloc[0]
            ).days
        else:
            row["history_span_days"] = 0

        X_rows.append(row)
        y.append(float(target_visit["mmse"]))
        patient_ids.append(patient_id)

    X = pd.DataFrame(X_rows)

    return (
        X,
        np.asarray(y, dtype=float),
        np.asarray(patient_ids)
    )


In [ ]:
X_train, y_train, train_patient_ids = (
    create_tabular_history_features(
        train_df,
        final_feature_cols
    )
)

X_val, y_val, val_patient_ids = (
    create_tabular_history_features(
        val_df,
        final_feature_cols
    )
)

X_test, y_test, test_patient_ids = (
    create_tabular_history_features(
        test_df,
        final_feature_cols
    )
)

# 保證三個 split 欄位順序完全一致
X_val = X_val.reindex(columns=X_train.columns)
X_test = X_test.reindex(columns=X_train.columns)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("Train patients:", len(train_patient_ids))
print("Val patients  :", len(val_patient_ids))
print("Test patients :", len(test_patient_ids))


### 模型與 validation tuning

三個模型都只用 validation set 選超參數，**test set 不參與選模**。
選好參數後，再使用 `train + validation` 重新 fit，最後只在 test set 評估一次。

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def tune_on_validation(
    estimator_class,
    param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    fixed_params=None
):
    """
    使用 validation RMSE 選擇最佳超參數。
    """
    fixed_params = fixed_params or {}

    tuning_rows = []
    best_rmse = np.inf
    best_params = None

    for params in ParameterGrid(param_grid):

        model = estimator_class(
            **fixed_params,
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        val_pred = model.predict(
            X_val
        )

        metrics = regression_metrics(
            y_val,
            val_pred
        )

        tuning_rows.append({
            **params,
            **metrics
        })

        if metrics["RMSE"] < best_rmse:
            best_rmse = metrics["RMSE"]
            best_params = params.copy()

    tuning_df = (
        pd.DataFrame(tuning_rows)
        .sort_values("RMSE")
        .reset_index(drop=True)
    )

    return best_params, tuning_df


### 1. SVM（SVR）

In [ ]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.1, 0.5, 1.0],
    "gamma": ["scale", 0.01, 0.1]
}

best_svm_params, svm_tuning = tune_on_validation(
    estimator_class=SVR,
    param_grid=svm_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "kernel": "rbf"
    }
)

print("Best SVM params:")
print(best_svm_params)

display(
    svm_tuning.head(10)
)


### 2. Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5]
}

best_rf_params, rf_tuning = tune_on_validation(
    estimator_class=RandomForestRegressor,
    param_grid=rf_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "random_state": SEED,
        "n_jobs": -1
    }
)

print("Best RF params:")
print(best_rf_params)

display(
    rf_tuning.head(10)
)


### 3. XGBoost

In [ ]:
xgb_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0]
}

best_xgb_params, xgb_tuning = tune_on_validation(
    estimator_class=XGBRegressor,
    param_grid=xgb_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "objective": "reg:squarederror",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0
    }
)

print("Best XGBoost params:")
print(best_xgb_params)

display(
    xgb_tuning.head(10)
)


### 使用最佳參數重新訓練，並在 test set 比較

Validation 已完成選模後，將 train + validation 合併重新訓練。
Test set 在這之前完全沒有參與 tuning。

In [ ]:
X_trainval = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_trainval = np.concatenate(
    [y_train, y_val]
)


svm_model = SVR(
    kernel="rbf",
    **best_svm_params
)

rf_model = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
    **best_rf_params
)

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
    **best_xgb_params
)


models = {
    "SVM": svm_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

test_predictions = {}
result_rows = []

for model_name, model in models.items():

    model.fit(
        X_trainval,
        y_trainval
    )

    y_pred = model.predict(
        X_test
    )

    test_predictions[model_name] = y_pred

    metrics = regression_metrics(
        y_test,
        y_pred
    )

    result_rows.append({
        "Model": model_name,
        **metrics
    })


results_df = (
    pd.DataFrame(result_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(results_df)


In [ ]:
# ============================================================
# Feature importance：Permutation Importance
# ============================================================
# 使用同一種 permutation importance 計算 SVM / RF / XGBoost，
# 因此三種模型的重要性可用一致定義比較。
# scoring 使用 negative RMSE；importance > 0 代表打亂該特徵會使預測變差。

feature_names = list(X_test.columns)

group_importance_rows = []

for model_name, model in models.items():
    perm = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="neg_root_mean_squared_error",
        n_repeats=20,
        random_state=SEED,
        n_jobs=-1
    )

    temp_imp = pd.DataFrame({
        "Group": GROUP,
        "Model": model_name,
        "Feature": feature_names,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std,
        "Method": "Permutation importance (test RMSE)"
    })

    temp_imp = temp_imp.sort_values(
        "Importance", ascending=False
    ).reset_index(drop=True)
    temp_imp["Rank"] = np.arange(1, len(temp_imp) + 1)

    group_importance_rows.append(temp_imp)
    all_feature_importance.append(temp_imp)

    print(f"\n=== {GROUP} | {model_name} | Top 20 features ===")
    display(temp_imp.head(20))

group_feature_importance = pd.concat(
    group_importance_rows, ignore_index=True
)


In [ ]:
# 每位 test patient 的實際值與三個模型預測值
prediction_df = pd.DataFrame({
    "patient_id": test_patient_ids,
    "y_true": y_test,
    "SVM_pred": test_predictions["SVM"],
    "RF_pred": test_predictions["Random Forest"],
    "XGBoost_pred": test_predictions["XGBoost"]
})

display(
    prediction_df.head()
)


In [ ]:
# Actual vs Predicted
for model_name, y_pred in test_predictions.items():

    plt.figure(figsize=(6, 6))

    plt.scatter(
        y_test,
        y_pred,
        alpha=0.7
    )

    lower = min(
        y_test.min(),
        y_pred.min()
    )
    upper = max(
        y_test.max(),
        y_pred.max()
    )

    plt.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--"
    )

    plt.xlabel("True MMSE")
    plt.ylabel("Predicted MMSE")
    plt.title(f"{model_name}: True vs Predicted")
    plt.show()


### Last-MMSE baseline

In [ ]:
def last_mmse_baseline(
    data,
    patient_ids
):

    predictions = []
    targets = []

    for patient_id in patient_ids:

        patient_df = (
            data[
                data["patient_id"]
                == patient_id
            ]
            .sort_values(
                "visit_date"
            )
        )

        if len(patient_df) < 2:
            continue

        previous_mmse = (
            patient_df.iloc[-2]["mmse"]
        )

        target_mmse = (
            patient_df.iloc[-1]["mmse"]
        )

        predictions.append(
            previous_mmse
        )

        targets.append(
            target_mmse
        )

    return (
        np.array(predictions),
        np.array(targets)
    )

In [ ]:
baseline_pred, baseline_true = (
    last_mmse_baseline(
        test_df,
        test_ids
    )
)

baseline_mae = mean_absolute_error(
    baseline_true,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_true,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    baseline_true,
    baseline_pred
)

print("Last-MMSE baseline")

print(
    f"MAE : {baseline_mae:.3f}"
)

print(
    f"RMSE: {baseline_rmse:.3f}"
)

print(
    f"R²  : {baseline_r2:.3f}"
)

In [ ]:
# 儲存此組別的 test 結果
group_result = results_df.copy()
group_result.insert(0, "Group", GROUP)

baseline_row = pd.DataFrame([{
    "Group": GROUP,
    "Model": "Last-MMSE baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
}])
# 對齊實際 metric 欄名（R2 或 R²）
if "R²" in group_result.columns and "R2" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R2": "R²"})
elif "R2" in group_result.columns and "R²" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R²": "R2"})

all_group_results.append(pd.concat([group_result, baseline_row], ignore_index=True))

group_pred = prediction_df.copy()
group_pred.insert(0, "Group", GROUP)
all_group_predictions.append(group_pred)


# Group: non-AD


In [ ]:
from sklearn.model_selection import train_test_split

GROUP = "non-AD"

group_df = df_model[
    df_model["bio_category"] == GROUP
].copy()

print(group_df.shape)

print(
    "Patients:",
    group_df["patient_id"].nunique()
)

#Patient-level train/validation/test split
patient_ids = (
    group_df["patient_id"]
    .drop_duplicates()
    .to_numpy()
)

train_ids, temp_ids = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=SEED
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=SEED
)

print("Train patients:", len(train_ids))
print("Val patients:", len(val_ids))
print("Test patients:", len(test_ids))

print(
    set(train_ids)
    & set(val_ids)
)

print(
    set(train_ids)
    & set(test_ids)
)

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

### 建立missing indicators

In [ ]:
train_df = group_df[
    group_df["patient_id"].isin(train_ids)
].copy()

val_df = group_df[
    group_df["patient_id"].isin(val_ids)
].copy()

test_df = group_df[
    group_df["patient_id"].isin(test_ids)
].copy()

missing_indicator_cols = []

for col in feature_cols:

    if train_df[col].isna().any():

        indicator_col = col + "_missing"

        train_df[indicator_col] = (
            train_df[col].isna().astype(int)
        )

        val_df[indicator_col] = (
            val_df[col].isna().astype(int)
        )

        test_df[indicator_col] = (
            test_df[col].isna().astype(int)
        )

        missing_indicator_cols.append(
            indicator_col
        )

In [ ]:
print(
    "Missing indicators:",
    len(missing_indicator_cols)
)

### 缺值處理 (中位數填補)

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)

imputer.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = imputer.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = imputer.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = imputer.transform(
    test_df[feature_cols]
)

In [ ]:
final_feature_cols = (
    feature_cols
    + missing_indicator_cols
)

### StandardScaler

SVM 對特徵尺度非常敏感，因此 numeric features 使用 train set fit 的 StandardScaler。
Random Forest / XGBoost 對尺度不敏感，但為了讓三個模型共用完全相同的輸入資料，
這裡仍使用同一份標準化後的資料；missing indicator 保持 0/1，不做 scaling。

In [ ]:
scaler = StandardScaler()
# missing indicator 不需要 scaling。所以只 scale 原始數值：
scaler.fit(
    train_df[feature_cols]
)

train_df[feature_cols] = scaler.transform(
    train_df[feature_cols]
)

val_df[feature_cols] = scaler.transform(
    val_df[feature_cols]
)

test_df[feature_cols] = scaler.transform(
    test_df[feature_cols]
)

### 將 longitudinal history 轉成固定長度特徵

SVM、Random Forest、XGBoost 無法直接處理每位病人長度不同的 sequence，
因此對「target 前的所有歷史 visits」建立固定長度 summary features：

- `last`：最近一次歷史 visit
- `mean`：歷史平均
- `std`：歷史標準差
- `min` / `max`：歷史範圍
- `history_n_visits`：target 前共有幾次 visit
- `history_span_days`：第一個歷史 visit 到最近歷史 visit 的天數

**Target 仍然是該病人的最後一次 MMSE，因此 test target 沒有被拿來建立輸入特徵。**

In [ ]:
def create_tabular_history_features(
    data,
    feature_cols
):
    """
    將每位 patient 在 target visit 之前的 longitudinal history
    轉成固定長度向量，供 SVM / RF / XGBoost 使用。

    Target = 該 patient 最後一次 visit 的 MMSE
    History = 最後一次 visit 之前的所有 visits
    """

    X_rows = []
    y = []
    patient_ids = []

    for patient_id, patient_df in data.groupby("patient_id"):

        patient_df = (
            patient_df
            .sort_values("visit_date")
            .reset_index(drop=True)
        )

        # 至少要有 2 次 visit：
        # 前面 >= 1 次做 history，最後 1 次做 target
        if len(patient_df) < 2:
            continue

        history = patient_df.iloc[:-1].copy()
        target_visit = patient_df.iloc[-1]

        hist_values = history[feature_cols]

        row = {}

        # 最近一次歷史 visit
        last_values = hist_values.iloc[-1]
        for col in feature_cols:
            row[f"{col}_last"] = last_values[col]

        # 使用全部歷史 visit 的 summary
        means = hist_values.mean(axis=0)
        stds = hist_values.std(axis=0, ddof=0)
        mins = hist_values.min(axis=0)
        maxs = hist_values.max(axis=0)

        for col in feature_cols:
            row[f"{col}_mean"] = means[col]
            row[f"{col}_std"] = stds[col]
            row[f"{col}_min"] = mins[col]
            row[f"{col}_max"] = maxs[col]

        row["history_n_visits"] = len(history)

        if len(history) >= 2:
            row["history_span_days"] = (
                history["visit_date"].iloc[-1]
                - history["visit_date"].iloc[0]
            ).days
        else:
            row["history_span_days"] = 0

        X_rows.append(row)
        y.append(float(target_visit["mmse"]))
        patient_ids.append(patient_id)

    X = pd.DataFrame(X_rows)

    return (
        X,
        np.asarray(y, dtype=float),
        np.asarray(patient_ids)
    )


In [ ]:
X_train, y_train, train_patient_ids = (
    create_tabular_history_features(
        train_df,
        final_feature_cols
    )
)

X_val, y_val, val_patient_ids = (
    create_tabular_history_features(
        val_df,
        final_feature_cols
    )
)

X_test, y_test, test_patient_ids = (
    create_tabular_history_features(
        test_df,
        final_feature_cols
    )
)

# 保證三個 split 欄位順序完全一致
X_val = X_val.reindex(columns=X_train.columns)
X_test = X_test.reindex(columns=X_train.columns)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("Train patients:", len(train_patient_ids))
print("Val patients  :", len(val_patient_ids))
print("Test patients :", len(test_patient_ids))


### 模型與 validation tuning

三個模型都只用 validation set 選超參數，**test set 不參與選模**。
選好參數後，再使用 `train + validation` 重新 fit，最後只在 test set 評估一次。

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def tune_on_validation(
    estimator_class,
    param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    fixed_params=None
):
    """
    使用 validation RMSE 選擇最佳超參數。
    """
    fixed_params = fixed_params or {}

    tuning_rows = []
    best_rmse = np.inf
    best_params = None

    for params in ParameterGrid(param_grid):

        model = estimator_class(
            **fixed_params,
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        val_pred = model.predict(
            X_val
        )

        metrics = regression_metrics(
            y_val,
            val_pred
        )

        tuning_rows.append({
            **params,
            **metrics
        })

        if metrics["RMSE"] < best_rmse:
            best_rmse = metrics["RMSE"]
            best_params = params.copy()

    tuning_df = (
        pd.DataFrame(tuning_rows)
        .sort_values("RMSE")
        .reset_index(drop=True)
    )

    return best_params, tuning_df


### 1. SVM（SVR）

In [ ]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.1, 0.5, 1.0],
    "gamma": ["scale", 0.01, 0.1]
}

best_svm_params, svm_tuning = tune_on_validation(
    estimator_class=SVR,
    param_grid=svm_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "kernel": "rbf"
    }
)

print("Best SVM params:")
print(best_svm_params)

display(
    svm_tuning.head(10)
)


### 2. Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5]
}

best_rf_params, rf_tuning = tune_on_validation(
    estimator_class=RandomForestRegressor,
    param_grid=rf_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "random_state": SEED,
        "n_jobs": -1
    }
)

print("Best RF params:")
print(best_rf_params)

display(
    rf_tuning.head(10)
)


### 3. XGBoost

In [ ]:
xgb_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0]
}

best_xgb_params, xgb_tuning = tune_on_validation(
    estimator_class=XGBRegressor,
    param_grid=xgb_param_grid,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    fixed_params={
        "objective": "reg:squarederror",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0
    }
)

print("Best XGBoost params:")
print(best_xgb_params)

display(
    xgb_tuning.head(10)
)


### 使用最佳參數重新訓練，並在 test set 比較

Validation 已完成選模後，將 train + validation 合併重新訓練。
Test set 在這之前完全沒有參與 tuning。

In [ ]:
X_trainval = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_trainval = np.concatenate(
    [y_train, y_val]
)


svm_model = SVR(
    kernel="rbf",
    **best_svm_params
)

rf_model = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
    **best_rf_params
)

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
    **best_xgb_params
)


models = {
    "SVM": svm_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

test_predictions = {}
result_rows = []

for model_name, model in models.items():

    model.fit(
        X_trainval,
        y_trainval
    )

    y_pred = model.predict(
        X_test
    )

    test_predictions[model_name] = y_pred

    metrics = regression_metrics(
        y_test,
        y_pred
    )

    result_rows.append({
        "Model": model_name,
        **metrics
    })


results_df = (
    pd.DataFrame(result_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(results_df)


In [ ]:
# ============================================================
# Feature importance：Permutation Importance
# ============================================================
# 使用同一種 permutation importance 計算 SVM / RF / XGBoost，
# 因此三種模型的重要性可用一致定義比較。
# scoring 使用 negative RMSE；importance > 0 代表打亂該特徵會使預測變差。

feature_names = list(X_test.columns)

group_importance_rows = []

for model_name, model in models.items():
    perm = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="neg_root_mean_squared_error",
        n_repeats=20,
        random_state=SEED,
        n_jobs=-1
    )

    temp_imp = pd.DataFrame({
        "Group": GROUP,
        "Model": model_name,
        "Feature": feature_names,
        "Importance": perm.importances_mean,
        "Importance_SD": perm.importances_std,
        "Method": "Permutation importance (test RMSE)"
    })

    temp_imp = temp_imp.sort_values(
        "Importance", ascending=False
    ).reset_index(drop=True)
    temp_imp["Rank"] = np.arange(1, len(temp_imp) + 1)

    group_importance_rows.append(temp_imp)
    all_feature_importance.append(temp_imp)

    print(f"\n=== {GROUP} | {model_name} | Top 20 features ===")
    display(temp_imp.head(20))

group_feature_importance = pd.concat(
    group_importance_rows, ignore_index=True
)


In [ ]:
# 每位 test patient 的實際值與三個模型預測值
prediction_df = pd.DataFrame({
    "patient_id": test_patient_ids,
    "y_true": y_test,
    "SVM_pred": test_predictions["SVM"],
    "RF_pred": test_predictions["Random Forest"],
    "XGBoost_pred": test_predictions["XGBoost"]
})

display(
    prediction_df.head()
)


In [ ]:
# Actual vs Predicted
for model_name, y_pred in test_predictions.items():

    plt.figure(figsize=(6, 6))

    plt.scatter(
        y_test,
        y_pred,
        alpha=0.7
    )

    lower = min(
        y_test.min(),
        y_pred.min()
    )
    upper = max(
        y_test.max(),
        y_pred.max()
    )

    plt.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--"
    )

    plt.xlabel("True MMSE")
    plt.ylabel("Predicted MMSE")
    plt.title(f"{model_name}: True vs Predicted")
    plt.show()


### Last-MMSE baseline

In [ ]:
def last_mmse_baseline(
    data,
    patient_ids
):

    predictions = []
    targets = []

    for patient_id in patient_ids:

        patient_df = (
            data[
                data["patient_id"]
                == patient_id
            ]
            .sort_values(
                "visit_date"
            )
        )

        if len(patient_df) < 2:
            continue

        previous_mmse = (
            patient_df.iloc[-2]["mmse"]
        )

        target_mmse = (
            patient_df.iloc[-1]["mmse"]
        )

        predictions.append(
            previous_mmse
        )

        targets.append(
            target_mmse
        )

    return (
        np.array(predictions),
        np.array(targets)
    )

In [ ]:
baseline_pred, baseline_true = (
    last_mmse_baseline(
        test_df,
        test_ids
    )
)

baseline_mae = mean_absolute_error(
    baseline_true,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_true,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    baseline_true,
    baseline_pred
)

print("Last-MMSE baseline")

print(
    f"MAE : {baseline_mae:.3f}"
)

print(
    f"RMSE: {baseline_rmse:.3f}"
)

print(
    f"R²  : {baseline_r2:.3f}"
)

In [ ]:
# 儲存此組別的 test 結果
group_result = results_df.copy()
group_result.insert(0, "Group", GROUP)

baseline_row = pd.DataFrame([{
    "Group": GROUP,
    "Model": "Last-MMSE baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
}])
# 對齊實際 metric 欄名（R2 或 R²）
if "R²" in group_result.columns and "R2" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R2": "R²"})
elif "R2" in group_result.columns and "R²" in baseline_row.columns:
    baseline_row = baseline_row.rename(columns={"R²": "R2"})

all_group_results.append(pd.concat([group_result, baseline_row], ignore_index=True))

group_pred = prediction_df.copy()
group_pred.insert(0, "Group", GROUP)
all_group_predictions.append(group_pred)


# Final summary


In [ ]:
# ============================================================
# CU / AD / non-AD 最終結果總結
# ============================================================
summary_results = pd.concat(all_group_results, ignore_index=True)
summary_results = summary_results.sort_values(["Group", "RMSE"]).reset_index(drop=True)

print("=== All groups: test performance ===")
display(summary_results)

# 每個組別 RMSE 最佳模型
best_by_group = (
    summary_results
    .sort_values("RMSE")
    .groupby("Group", as_index=False)
    .first()
)
print("=== Best model by group (lowest RMSE) ===")
display(best_by_group)

# 方便論文報告的 pivot table
rmse_table = summary_results.pivot(index="Group", columns="Model", values="RMSE")
mae_table = summary_results.pivot(index="Group", columns="Model", values="MAE")
print("=== RMSE comparison ===")
display(rmse_table)
print("=== MAE comparison ===")
display(mae_table)

# 合併所有 test patient predictions
all_predictions_df = pd.concat(all_group_predictions, ignore_index=True)
display(all_predictions_df.head())

# 可選：輸出 CSV
summary_results.to_csv("CU_AD_nonAD_model_summary.csv", index=False)
all_predictions_df.to_csv("CU_AD_nonAD_test_predictions.csv", index=False)


In [ ]:
# ============================================================
# Feature importance 最終總結與 CSV 輸出
# ============================================================
feature_importance_summary = pd.concat(
    all_feature_importance,
    ignore_index=True
)

feature_importance_summary = feature_importance_summary.sort_values(
    ["Group", "Model", "Rank"]
).reset_index(drop=True)

print("=== CU / AD / non-AD：全部模型 Feature Importance ===")
display(feature_importance_summary)

# 每個 Group × Model 的 Top 20，方便快速檢查
top20_feature_importance = (
    feature_importance_summary
    .query("Rank <= 20")
    .copy()
)

print("=== Top 20 features by Group × Model ===")
display(top20_feature_importance)

# 完整版：保留所有 features
feature_importance_summary.to_csv(
    "CU_AD_nonAD_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

# 另外輸出 Top 20 版本
top20_feature_importance.to_csv(
    "CU_AD_nonAD_feature_importance_top20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: CU_AD_nonAD_feature_importance.csv")
print("Saved: CU_AD_nonAD_feature_importance_top20.csv")
